## 1. Import thư viện

In [2]:
import json
import faiss
import torch
import numpy as np
import pandas as pd
import torch.nn.functional as F
from pathlib import Path

from transformers import AutoTokenizer, AutoModel

d:\anaconda3\envs\it\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Config

In [3]:
OUTPUT_DIR = Path("data/e5_base")
MODEL_NAME = "intfloat/multilingual-e5-base"
MAX_LENGTH = 512
TEXT_COL = "raw_text"

## 3. Load Model (Dùng để encode query online)

In [4]:
device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.float16 if device == "cuda" else torch.float32

print("device:", device)
print("dtype:", dtype)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModel.from_pretrained(
    MODEL_NAME,
    torch_dtype=dtype,
).to(device)

model.eval()

device: cuda
dtype: torch.float16


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 351.25it/s]


XLMRobertaModel(
  (embeddings): XLMRobertaEmbeddings(
    (word_embeddings): Embedding(250002, 768, padding_idx=1)
    (token_type_embeddings): Embedding(1, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
    (position_embeddings): Embedding(514, 768, padding_idx=1)
  )
  (encoder): XLMRobertaEncoder(
    (layer): ModuleList(
      (0-11): 12 x XLMRobertaLayer(
        (attention): XLMRobertaAttention(
          (self): XLMRobertaSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): XLMRobertaSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=Tru

## 4. Hàm encode chuẩn cho E5

### Mean pooling token embeddings

Cell này định nghĩa cách gom embedding của từng token thành một vector đại diện cho cả đoạn text. `attention_mask` giúp bỏ qua padding token khi tính trung bình.

In [5]:
def mean_pooling(last_hidden_state, attention_mask):
    mask = attention_mask.unsqueeze(-1).expand(last_hidden_state.size()).float()
    summed = torch.sum(last_hidden_state.float() * mask, dim=1)
    counts = torch.clamp(mask.sum(dim=1), min=1e-9)
    return summed / counts

### Hàm encode danh sách văn bản

Cell này biến danh sách text thành ma trận embedding. Mỗi batch sẽ được tokenize, đưa qua model, mean pooling, normalize L2 và trả về `float32` để FAISS có thể index/search ổn định.

In [6]:
@torch.no_grad()
def encode_texts(texts, batch_size=1, max_length=MAX_LENGTH):
    all_embeddings = []

    for start in range(0, len(texts), batch_size):
        batch_texts = texts[start:start + batch_size]

        inputs = tokenizer(
            batch_texts,
            max_length=max_length,
            padding=True,
            truncation=True,
            return_tensors="pt",
        )

        inputs = {k: v.to(device) for k, v in inputs.items()}

        with torch.amp.autocast("cuda", enabled=(device == "cuda")):
            outputs = model(**inputs)

        embeddings = mean_pooling(
            outputs.last_hidden_state,
            inputs["attention_mask"],
        )

        embeddings = F.normalize(embeddings, p=2, dim=1)

        all_embeddings.append(
            embeddings.cpu().numpy().astype("float32")
        )

        del inputs, outputs, embeddings

        if device == "cuda":
            torch.cuda.empty_cache()

    return np.vstack(all_embeddings)

## 5. Semantic Search Helpers

### Hàm load artifact semantic đã lưu

Cell này đóng gói việc load FAISS index, metadata và config từ `OUTPUT_DIR`. Hàm này hữu ích khi mở lại notebook và chỉ muốn search, không muốn chạy lại bước tạo embedding.

In [7]:
def load_semantic_artifacts(output_dir=OUTPUT_DIR):
    output_dir = Path(output_dir)
    index_path = output_dir / "semantic_e5_base.faiss"
    metadata_path = output_dir / "chunks_metadata.parquet"
    config_path = output_dir / "embedding_config.json"

    loaded_index = faiss.read_index(str(index_path))
    loaded_metadata = pd.read_parquet(metadata_path)

    with open(config_path, "r", encoding="utf-8") as f:
        loaded_config = json.load(f)

    print("index vectors:", loaded_index.ntotal)
    print("metadata rows:", len(loaded_metadata))
    print("model:", loaded_config["model_name"])

    return loaded_index, loaded_metadata, loaded_config

### Nạp artifact vào biến sử dụng

Cell này gọi `load_semantic_artifacts()` và gán kết quả vào `index`, `metadata`, `semantic_config` để các hàm search phía sau dùng chung.

In [8]:
index, metadata, semantic_config = load_semantic_artifacts()

index vectors: 1078604
metadata rows: 1078604
model: intfloat/multilingual-e5-base


### Hàm semantic search nâng cấp

Cell này mở rộng hàm search cơ bản: đọc prefix/max length từ config, lấy nhiều ứng viên hơn bằng `candidate_k`, hỗ trợ lọc theo `topic`, lọc theo `min_score`, và trả về DataFrame đã sắp xếp theo score.

In [9]:
def semantic_search_v2(
    query,
    top_k=10,
    candidate_k=None,
    topic=None,
    min_score=None,
):
    if not query or not str(query).strip():
        return pd.DataFrame()

    candidate_k = candidate_k or max(top_k * 5, 50)
    candidate_k = min(candidate_k, index.ntotal)

    query_prefix = semantic_config.get("query_prefix", "query: ")
    query_text = query_prefix + str(query).strip()

    query_embedding = encode_texts(
        [query_text],
        batch_size=1,
        max_length=semantic_config.get("max_length", MAX_LENGTH),
    )

    scores, ids = index.search(query_embedding, candidate_k)
    rows = []

    for score, idx in zip(scores[0], ids[0]):
        if idx == -1:
            continue
        if min_score is not None and score < min_score:
            continue

        row = metadata.iloc[int(idx)].to_dict()
        if topic is not None and row.get("topic") != topic:
            continue

        row["score"] = float(score)
        rows.append(row)

        if len(rows) >= top_k:
            break

    return pd.DataFrame(rows)

### Định dạng kết quả hiển thị

Cell này cắt ngắn cột `text` và chỉ giữ các cột quan trọng như score, doc_id, chunk_id, topic, text, url. Mục đích là làm bảng kết quả dễ đọc trong notebook.

In [10]:
def show_search_results(results, max_text_chars=220):
    if results.empty:
        return results

    display_df = results.copy()
    display_df["text"] = display_df["text"].str.slice(0, max_text_chars)

    columns = [
        "score",
        "doc_id",
        "chunk_id",
        "topic",
        "text",
        "url",
    ]
    columns = [col for col in columns if col in display_df.columns]
    return display_df[columns]

### Chạy thử hàm search nâng cấp

Cell này gọi `semantic_search_v2` với query mẫu và đưa kết quả qua `show_search_results()` để xem bảng gọn, dễ kiểm tra nhanh chất lượng search.

In [11]:
results = semantic_search_v2("cướp tiệm vàng", top_k=5)
show_search_results(results)

,score,doc_id,chunk_id,topic,text,url
0,0.898489,1336,1336_6,Pháp luật,"Tên cướp tiệm vàng tại Huế là đại uý công an, ...",https://docbao.vn/phap-luat/ten-cuop-tiem-vang...
1,0.898419,0,0_6,Pháp luật,"Tên cướp tiệm vàng tại Huế là đại uý công an, ...",https://docbao.vn/phap-luat/ten-cuop-tiem-vang...
2,0.896942,997,997_6,Pháp luật,Phó giám đốc Công an kể phút đấu trí với nghi ...,https://docbao.vn/phap-luat/pho-giam-doc-cong-...
3,0.894356,14478,14478_2,Thời sự,"Thiếu tiền tiêu, nam thanh niên đi cướp tiệm v...",https://thanhnien.vn/thieu-tien-tieu-nam-thanh...
4,0.890606,1158,1158_0,Thời sự,Nghi phạm cướp tiệm vàng ở Huế vứt vàng ra đườ...,https://vietnamnet.vn/nghi-pham-cuop-tiem-vang...


## 6. Gộp kết quả theo bài viết

### Gộp kết quả theo bài viết

Cell này xử lý trường hợp một bài viết có nhiều chunk cùng lọt top. Hàm giữ chunk có score cao nhất cho mỗi `doc_id`, giúp màn hình kết quả đa dạng hơn theo bài viết.

In [12]:
def group_results_by_doc(results, top_k=5):
    if results.empty:
        return results

    grouped = (
        results.sort_values("score", ascending=False)
        .groupby("doc_id", as_index=False)
        .agg(
            score=("score", "max"),
            chunk_id=("chunk_id", "first"),
            topic=("topic", "first"),
            text=("text", "first"),
            url=("url", "first"),
        )
        .sort_values("score", ascending=False)
        .head(top_k)
    )

    return grouped.reset_index(drop=True)

### Ví dụ search và gộp theo bài viết

Cell này lấy nhiều chunk ứng viên, gộp lại theo `doc_id`, rồi hiển thị top bài viết. Cách này gần với UI search hơn vì người dùng thường muốn xem bài viết, không phải các chunk trùng nhau.

In [13]:
chunk_results = semantic_search_v2("quân đội Trung Quốc Thái Bình Dương", top_k=20)
doc_results = group_results_by_doc(chunk_results, top_k=5)
show_search_results(doc_results)

,score,doc_id,chunk_id,topic,text,url
0,0.877806,25764,25764_0,Thế giới,Mỹ cảnh báo về quân đội Trung Quốc | Báo Dân t...,https://dantri.com.vn/the-gioi/my-canh-bao-ve-...
1,0.872181,146818,146818_0,Thế giới,Trung Quốc tuyên bố mở rộng hoạt động quân sự ...,https://baoquocte.vn/trung-quoc-tuyen-bo-mo-ro...
2,0.866188,24870,24870_3,Thế giới,Tướng Mỹ: 'Trung Quốc ngày càng hung hăng ở Th...,https://tuoitre.vn/tuong-my-trung-quoc-ngay-ca...
3,0.863620,144393,144393_2,Thế giới,Chuyên gia phân tích lợi thế quân sự của Mỹ và...,https://www.24h.com.vn/tin-tuc-quoc-te/chuyen-...
4,0.861696,150052,150052_5,Thế giới,Ông Tập Cận Bình ký sắc lệnh cho phép quân đội...,https://tienphong.vn/ong-tap-can-binh-ky-sac-l...


Ghi chú: E5 cần prefix `passage: ` khi encode document và `query: ` khi encode câu hỏi. Vì embedding đã normalize nên FAISS `IndexFlatIP` tương đương cosine similarity.

In [14]:
# Tạo thư mục lưu model E5 offline
local_e5_path = "models/e5_base_offline"
Path(local_e5_path).mkdir(parents=True, exist_ok=True)

# Lưu Tokenizer và Model
tokenizer.save_pretrained(local_e5_path)
model.save_pretrained(local_e5_path)

print(f"Đã lưu thành công Tokenizer và Model E5 tại: {local_e5_path}")

"""
Ghi chú cho lần chạy sau: 
Thay vì dùng MODEL_NAME = "intfloat/multilingual-e5-base", 
bạn có thể trỏ trực tiếp vào thư mục local này:
MODEL_NAME = "models/e5_base_offline"
"""

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.50it/s]

Đã lưu thành công Tokenizer và Model E5 tại: models/e5_base_offline


'\nGhi chú cho lần chạy sau: \nThay vì dùng MODEL_NAME = "intfloat/multilingual-e5-base", \nbạn có thể trỏ trực tiếp vào thư mục local này:\nMODEL_NAME = "models/e5_base_offline"\n'